In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{PerceptronLoss} = L(z, t) = \begin{cases} 0 & \text{if } zt > 0 \\ -zt & \text{if } zt \leq 0 \end{cases} $$

$$ \frac{\partial L}{\partial z} = \begin{cases} 0 & \text{if } zt > 0 \\ -t & \text{if } zt \leq 0 \end{cases} $$

In [ ]:
import torch as tr
import torch.nn as nn
import torch.autograd as autograd

import import_ipynb
from approx import approx # type: ignore

def ploss(z, t, reduction = "mean"):
    """Calculate the `perceptron loss` between predicted `z` logits and true targets `t`."""

    assert tr.all((t == -1.0) | (t == +1.0))

    mask = (z * t <= 0)
    loss = tr.zeros_like(z)
    loss[mask] = - z[mask] * t[mask]
    
    if reduction != "mean":
        assert False, f"Unsupported reduction: {reduction}"

    loss = loss.mean()
    return loss


class PLossFunction(autograd.Function):
    """Custom autograd function for the `perceptron loss`."""

    @staticmethod
    def forward(ctx, z, t):
        ctx.save_for_backward(z, t)

        loss = ploss(z, t)
        return loss

    @staticmethod
    def backward(ctx, grad_output):
        (z, t) = ctx.saved_tensors
        S = z.shape[0]
        FO = z.shape[1]
        N = S * FO

        mask = (z * t <= 0)
        df_dz = tr.zeros_like(z)
        df_dz[mask] = -t[mask]

        # Average over all elements
        df_dz = df_dz / N

        # For this example grad_output == 1
        df_dz = df_dz * grad_output
        assert df_dz.shape == (S, FO)
        
        return (df_dz, None)
    

class PLoss(nn.Module):
    """Custom module for the `perceptron loss`."""

    def __init__(self, reduction: str = "mean"):
        super().__init__()

        if reduction != "mean":
            assert False, f"Unsupported reduction: {reduction}"
    
    def forward(self, z: tr.Tensor, t: tr.Tensor) -> tr.Tensor:
        L = PLossFunction.apply(z, t)
        return L

In [ ]:
# Unit tests

def test_gradcheck(samples=10):
    def fn(z, t):
        return PLossFunction.apply(z, t)

    z = tr.randn(samples, 1, dtype=tr.float64, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0, dtype=tr.float64),
        tr.tensor(+1.0, dtype=tr.float64)
    )

    assert autograd.gradcheck(fn, (z, t), eps=1e-6, atol=1e-4, rtol=1e-4)


def test_compare(samples):
    z = tr.randn(samples, 1, requires_grad=True)

    t = tr.where(
        tr.rand(samples, 1) < 0.5,
        tr.tensor(-1.0),
        tr.tensor(+1.0)
    ).reshape(samples, 1)

    z1 = z.clone().detach().requires_grad_(True)
    y1 = PLoss()(z1, t)
    y1.backward()

    z2 = z.clone().detach().requires_grad_(True)
    y2 = tr.max(tr.zeros_like(z2), - z2 * t).mean()
    y2.backward()

    assert y1.item() == approx(y2.item())
    assert z1.grad == approx(z2.grad)


if __name__ == "__main__":
    test_gradcheck(1)
    test_gradcheck(10)
    test_gradcheck(100)

    test_compare(1)
    test_compare(10)
    test_compare(100)